In [16]:
import pandas as pd, plotly.express as pe 

In [17]:
import pandas as pd

train_tx = pd.read_csv("../data/train_transaction.csv")
train_id = pd.read_csv("../data/train_identity.csv")

print(train_tx.shape)
print(train_id.shape)
print(train_tx["isFraud"].value_counts(normalize=True))
print(train_tx.columns.tolist())

(590540, 394)
(144233, 41)
isFraud
0    0.96501
1    0.03499
Name: proportion, dtype: float64
['TransactionID', 'isFraud', 'TransactionDT', 'TransactionAmt', 'ProductCD', 'card1', 'card2', 'card3', 'card4', 'card5', 'card6', 'addr1', 'addr2', 'dist1', 'dist2', 'P_emaildomain', 'R_emaildomain', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C12', 'C13', 'C14', 'D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8', 'D9', 'D10', 'D11', 'D12', 'D13', 'D14', 'D15', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'V29', 'V30', 'V31', 'V32', 'V33', 'V34', 'V35', 'V36', 'V37', 'V38', 'V39', 'V40', 'V41', 'V42', 'V43', 'V44', 'V45', 'V46', 'V47', 'V48', 'V49', 'V50', 'V51', 'V52', 'V53', 'V54', 'V55', 'V56', 'V57', 'V58', 'V59', 'V60', 'V61', 'V62', 'V63', 'V64', 'V65', 'V66', 'V67', 'V68',

In [18]:
FEATURES = [
    # Transaction
    "TransactionAmt",
    "ProductCD",
    
    # Card info
    "card1", "card2", "card3", "card4", "card5", "card6",
    
    # Address
    "addr1", "addr2",
    
    # Distance
    "dist1",
    
    # Email domains
    "P_emaildomain", "R_emaildomain",
    
    # Count features (frequency encoding signals)
    "C1", "C2", "C3", "C4", "C5", "C6", "C7", "C8", "C9", "C10",
    
    # Time delta features
    "D1", "D2", "D3", "D4", "D5",
    
    # Match features (boolean-ish)
    "M1", "M2", "M3", "M4", "M5", "M6", "M7", "M8", "M9",
]

In [ ]:

import joblib
import numpy as np
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.preprocessing import LabelEncoder

FEATURES = [
    "TransactionAmt", "ProductCD",
    "card1", "card2", "card3", "card4", "card5", "card6",
    "addr1", "addr2", "dist1",
    "P_emaildomain", "R_emaildomain",
    "C1", "C2", "C3", "C4", "C5", "C6", "C7", "C8", "C9", "C10",
    "D1", "D2", "D3", "D4", "D5",
    "M1", "M2", "M3", "M4", "M5", "M6", "M7", "M8", "M9",
]

df = pd.read_csv("data/train_transaction.csv")
df = df[FEATURES + ["isFraud", "TransactionDT"]]

# Encode categoricals
cat_cols = ["ProductCD", "card4", "card6", "P_emaildomain", "R_emaildomain",
            "M1", "M2", "M3", "M4", "M5", "M6", "M7", "M8", "M9"]
for col in cat_cols:
    le = LabelEncoder()
    df[col] = df[col].astype(str)
    df[col] = le.fit_transform(df[col])

# Fill nulls
df.fillna(-999, inplace=True)

# Temporal split
split = int(df["TransactionDT"].quantile(0.8))
train = df[df["TransactionDT"] <= split]
test  = df[df["TransactionDT"] > split]

X_train, y_train = train[FEATURES], train["isFraud"]
X_test,  y_test  = test[FEATURES],  test["isFraud"]

scale = (y_train == 0).sum() / (y_train == 1).sum()
model = XGBClassifier(scale_pos_weight=scale, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}")

In [ ]:
joblib.dump(model, "models/ato_xgb.pkl")
print("Model saved.")